In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/Actual_generation_202212250000_202512250000_Hour.csv", sep=";")

In [3]:
df["Start date"] = pd.to_datetime(df["Start date"], format="%b %d, %Y %I:%M %p")

In [4]:
cols_to_convert = [
    "Photovoltaics [MWh] Calculated resolutions",
    "Wind offshore [MWh] Calculated resolutions",
    "Wind onshore [MWh] Calculated resolutions"
]

for col in cols_to_convert:
    # Remove any stray spaces and force conversion to float
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [5]:
df["Wind"] = (
    df["Wind offshore [MWh] Calculated resolutions"] + 
    df["Wind onshore [MWh] Calculated resolutions"]
)

In [6]:
solar_col = "Photovoltaics [MWh] Calculated resolutions"
wind_col = "Wind"

In [7]:
is_peak_hours = (df["Start date"].dt.hour >= 8) & (df["Start date"].dt.hour < 20)

In [8]:
df["Market Period"] = "Off-peak"
df.loc[is_peak_hours, "Market Period"] = "Peak"

In [9]:
df["Date"] = df["Start date"].dt.date

In [10]:
daily_results = []

for current_date, group in df.groupby("Date"):
    peak_group = group[group["Market Period"] == "Peak"]
    offpeak_group = group[group["Market Period"] == "Off-peak"]
    
    solar_base = group[solar_col].mean()
    solar_peak = peak_group[solar_col].mean() if not peak_group.empty else np.nan
    solar_offpeak = offpeak_group[solar_col].mean() if not offpeak_group.empty else np.nan
    
    wind_base = group[wind_col].mean()
    wind_peak = peak_group[wind_col].mean() if not peak_group.empty else np.nan
    wind_offpeak = offpeak_group[wind_col].mean() if not offpeak_group.empty else np.nan
    
    daily_results.append({
        "Date": current_date,
        "Solar Baseload [MWh]": round(solar_base, 2) if pd.notna(solar_base) else "",
        "Solar Peakload [MWh]": round(solar_peak, 2) if pd.notna(solar_peak) else "",
        "Solar Off-peak [MWh]": round(solar_offpeak, 2) if pd.notna(solar_offpeak) else "",
        "Wind Baseload [MWh]": round(wind_base, 2) if pd.notna(wind_base) else "",
        "Wind Peakload [MWh]": round(wind_peak, 2) if pd.notna(wind_peak) else "",
        "Wind Off-peak [MWh]": round(wind_offpeak, 2) if pd.notna(wind_offpeak) else ""
    })

In [11]:
result_df = pd.DataFrame(daily_results)
result_df.to_csv("../data/wind_solar_production_loads_last_week.csv", sep=";", index=False)